# 01. 이미지 수집 · 정제 (YOLOv8 과제)

이 노트북은 여러 이미지 출처에서 부류별 사진을 모으고, 깨진 파일·너무 작은 사진·중복 사진을 자동으로 걸러낸 뒤,
사람이 눈으로 검수할 수 있게 번호가 붙은 미리보기(contact sheet)를 만들어 줍니다.

**수집 경로** (셀 2의 `SOURCES`에서 하나씩 켜고 끌 수 있음)

| 경로 | 검색 방식 | 특징 |
| --- | --- | --- |
| iNaturalist | 학명 (`taxa`) | 연구 등급 관찰 사진. 야외에서 찍은 살아 있는 개체가 대부분이라 품질이 가장 좋음 |
| GBIF | 학명 (`taxa`) | 여러 관찰 플랫폼의 사진 모음. iNaturalist와 겹치는 사진은 자동으로 건너뜀 |
| Wikimedia Commons | 학명 + 영어 검색어 | 라이선스와 작성자가 명확함 |
| Openverse | 학명 + 영어 검색어 | CC 라이선스 이미지 검색 (Flickr 등) |
| Flickr | 학명 + 영어 검색어 | API 키가 있을 때만 (`FLICKR_API_KEY`) |
| 네이버 | 한국어 검색어 (`ko`) | 국내 양봉장 사진. 검색어 하나에 약 50장까지 |
| Bing · DuckDuckGo | 영어 검색어 (`en`) | 결과가 들쭉날쭉함 ("Vespa" → 스쿠터 등). 검수 필요 |
| Baidu | 중국어 검색어 (`zh`) | 중국·대만 사진 |

Google 크롤러는 현재 결과가 0장이라 넣지 않았고, Bing은 한국어·일본어·중국어 검색어에 결과를 주지 않아 영어 검색어에만 씁니다.

**사용 순서**
1. 셀 2에서 `TOPIC`과 `SOURCES`를 확인합니다 (기본값 `"hornets"` = 꿀벌·장수말벌·등검은말벌).
2. 위에서부터 차례로 실행합니다. 수집은 경로를 다 켜면 10~20분 걸립니다. 중간에 끊겨도 셀 3을 다시 실행하면 받은 파일은 건너뜁니다.
3. 미리보기에서 잘못된 사진(일러스트, 표본, 다른 동물, 글자 위주 이미지 등)의 번호를 셀 6의 `DELETE`에 적고 실행합니다.
4. 마지막 셀에서 `dataset_clean.zip`(Roboflow 업로드용)과 `sources.csv`(출처 기록, 보고서용)를 내려받습니다.

`hard_negative` 폴더는 **헷갈리기 쉬운 다른 곤충**(예: 꿀벌을 흉내 내는 꽃등에, 쌍살벌), `background` 폴더는 **동물이 없는 배경**입니다.
Roboflow에서 이 사진들은 박스를 치지 않고 **Null(빈 이미지)** 로 표시합니다. 모델이 이것들을 잘못 탐지하지 않도록 가르치는 용도입니다.

수집한 이미지는 저작권 문제로 저장소에 올리지 않고, 이미지마다 출처·라이선스·작성자를 `sources.csv`에 기록합니다.

In [ ]:
!pip -q install icrawler imagehash ddgs

In [ ]:
# ===== 셀 2: 설정 =====
TOPIC = "hornets"       # "hornets"(꿀벌·장수말벌·등검은말벌) / "invasive"(생태계 교란 생물) / "birds"(도심 새)

# 수집 경로: on = 사용 여부, max = 검색어(또는 학명) 하나당 최대 장수
SOURCES = {
    "inat":      {"on": True,  "max": 600},   # iNaturalist (학명)
    "gbif":      {"on": True,  "max": 400},   # GBIF (학명)
    "commons":   {"on": True,  "max": 150},   # Wikimedia Commons (학명 + 영어)
    "openverse": {"on": True,  "max": 200},   # Openverse (학명 + 영어)
    "flickr":    {"on": False, "max": 300},   # Flickr (학명 + 영어) — FLICKR_API_KEY가 있을 때 True로
    "naver":     {"on": True,  "max": 50},    # 네이버 (한국어, 검색어당 최대 약 50장)
    "bing":      {"on": True,  "max": 60},    # Bing (영어) — 무관한 사진(인물·스쿠터 등)이 많아 적게 받음
    "ddg":       {"on": True,  "max": 60},    # DuckDuckGo (영어) — 위와 같음. 미리보기가 쓰레기투성이면 끄기
    "baidu":     {"on": True,  "max": 100},   # Baidu (중국어)
}
FLICKR_API_KEY = ""     # https://www.flickr.com/services/api/misc.api_keys.html 에서 발급
NEG_RATIO = 0.3         # hard_negative·background는 max에 이 비율을 곱한 만큼만 받음
INAT_CC_ONLY = False    # True면 iNaturalist에서 CC 라이선스 사진만 받음
DOWNLOAD_THREADS = 16

# taxa = 학명 (iNaturalist·GBIF·Commons·Openverse·Flickr), en = 영어 검색어, ko = 한국어 검색어 (네이버), zh = 중국어 검색어 (Baidu)
PRESETS = {
    "hornets": {
        "classes": {
            "honeybee": {
                "taxa": ["Apis mellifera"],
                "en": ["honey bee Apis mellifera", "honeybees at hive entrance", "honey bee flying", "honey bee on flower", "honey bees on frame beekeeping"],
                "ko": ["양봉꿀벌", "서양꿀벌", "꿀벌 벌통 입구", "꿀벌 비행", "꿀벌 꽃", "양봉 꿀벌 소비"],
                "zh": ["西方蜜蜂", "意大利蜜蜂", "蜜蜂 蜂箱"],
            },
            "giant_hornet": {
                "taxa": ["Vespa mandarinia"],
                "en": ["Asian giant hornet", "Vespa mandarinia hornet", "giant hornet attacking beehive", "giant hornet apiary"],
                "ko": ["장수말벌", "장수말벌 사진", "장수말벌 양봉", "장수말벌 벌통 공격", "장수말벌 꿀벌"],
                "zh": ["金环胡蜂", "大虎头蜂", "金环胡蜂 蜂箱"],
            },
            "yellow_legged_hornet": {
                "taxa": ["Vespa velutina"],
                "en": ["Asian hornet Vespa velutina", "yellow-legged hornet Vespa velutina", "Asian hornet hawking beehive", "Vespa velutina nigrithorax"],
                "ko": ["등검은말벌", "등검은말벌 사진", "등검은말벌 양봉장", "등검은말벌 벌통", "등검은말벌 꿀벌 사냥"],
                "zh": ["墨胸胡蜂", "黄腿胡蜂", "墨胸胡蜂 蜂箱"],
            },
        },
        # 꿀벌·말벌과 헷갈리기 쉬운 곤충 → 박스 없이 Null로 업로드
        "hard_negative": {
            "taxa": ["Syrphidae", "Eristalis tenax", "Polistes", "Bombus", "Musca domestica"],
            "en": ["hoverfly on flower", "paper wasp Polistes", "bumblebee on flower", "housefly macro"],
            "ko": ["꽃등에", "쌍살벌", "호박벌", "집파리"],
            "zh": ["食蚜蝇", "马蜂", "熊蜂"],
        },
        "background": {
            "taxa": [],
            "en": ["empty wooden beehive box", "apiary landscape", "beehive boxes in apiary", "flower meadow close up"],
            "ko": ["빈 벌통", "양봉장 풍경", "양봉장 벌통", "꽃밭"],
            "zh": ["蜂箱", "养蜂场"],
        },
    },
    "invasive": {
        "classes": {
            "bullfrog": {"taxa": ["Lithobates catesbeianus"], "en": ["American bullfrog", "bullfrog in pond"], "ko": ["황소개구리"], "zh": ["牛蛙"]},
            "red_eared_slider": {"taxa": ["Trachemys scripta elegans"], "en": ["red-eared slider turtle", "red-eared slider basking on log"], "ko": ["붉은귀거북"], "zh": ["巴西龟"]},
            "nutria": {"taxa": ["Myocastor coypus"], "en": ["nutria coypu", "nutria swimming river"], "ko": ["뉴트리아"], "zh": ["河狸鼠"]},
        },
        # 생김새가 비슷한 토종·보호종 → 교란종으로 오탐하면 안 되는 대상
        "hard_negative": {"taxa": ["Lutra lutra", "Mauremys reevesii", "Pelodiscus sinensis", "Pelophylax nigromaculatus", "Ondatra zibethicus", "Castor"],
                          "en": ["Eurasian otter", "black-spotted pond frog"], "ko": ["수달", "남생이", "자라", "참개구리"], "zh": []},
        "background": {"taxa": [], "en": ["pond water surface", "river bank reeds", "wetland stream rocks"], "ko": ["저수지 물가", "하천 갈대"], "zh": []},
    },
    "birds": {
        "classes": {
            "magpie": {"taxa": ["Pica serica"], "en": ["Oriental magpie", "magpie flying"], "ko": ["까치"], "zh": ["喜鹊"]},
            "pigeon": {"taxa": ["Columba livia"], "en": ["feral pigeon", "pigeons flock street"], "ko": ["비둘기"], "zh": ["鸽子"]},
            "sparrow": {"taxa": ["Passer montanus"], "en": ["Eurasian tree sparrow", "sparrow flying"], "ko": ["참새"], "zh": ["麻雀"]},
        },
        "hard_negative": {"taxa": ["Corvus macrorhynchos", "Garrulus glandarius", "Motacilla alba"], "en": ["large-billed crow"], "ko": ["큰부리까마귀", "어치", "백할미새"], "zh": []},
        "background": {"taxa": [], "en": ["empty city park", "tree branches sky", "empty sidewalk"], "ko": ["빈 공원", "나뭇가지 하늘"], "zh": []},
    },
}

CFG = PRESETS[TOPIC]
GROUPS = {**CFG["classes"], "hard_negative": CFG["hard_negative"], "background": CFG["background"]}
RAW, CLEAN, REVIEW = "/content/raw", "/content/clean", "/content/review"
print("TOPIC:", TOPIC, "| 부류:", list(CFG["classes"]))
print("사용 경로:", [s for s, c in SOURCES.items() if c["on"]])

In [ ]:
# ===== 셀 3: 수집 (여러 경로) =====
# 다시 실행하면 이미 받은 파일은 건너뜁니다 (raw/manifest.json에 기록).
import os, re, json, time, html, logging, threading
import requests
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
API_UA = "BeeGuardStudentProject/0.2 (https://github.com/djkdb/beeguard-hornet-tracker)"   # Wikimedia 등 API 예절
HTTP = requests.Session()
HTTP.headers["User-Agent"] = UA
os.makedirs(RAW, exist_ok=True)
MANIFEST_PATH = os.path.join(RAW, "manifest.json")
MANIFEST = json.load(open(MANIFEST_PATH, encoding="utf-8")) if os.path.exists(MANIFEST_PATH) else {}
_lock = threading.Lock()

def get_json(url, params, headers=None, tries=2):
    """API 호출. 429·5xx면 잠시 쉬었다가 다시 시도한다."""
    for i in range(tries):
        try:
            r = HTTP.get(url, params=params, headers=headers, timeout=15)
            if r.status_code == 200:
                return r.json()
            if r.status_code not in (429, 500, 502, 503, 504):
                return {}
        except (requests.RequestException, ValueError):
            pass
        time.sleep(3 * (i + 1))
    return {}

def item(url, license="", author="", page=""):
    return {"url": url, "license": license or "", "author": (author or "").strip()[:200], "page": page or ""}

# ---------- 학명 기반 ----------
_taxon_cache = {}
def inat_taxon_id(name):
    if name not in _taxon_cache:
        res = get_json("https://api.inaturalist.org/v1/taxa", {"q": name, "per_page": 5}).get("results", [])
        exact = [t for t in res if t.get("name", "").lower() == name.lower()]
        _taxon_cache[name] = (exact or res or [{}])[0].get("id")
    return _taxon_cache[name]

def src_inat(q, n):
    tid = inat_taxon_id(q)
    if not tid:
        return []
    out, page = [], 1
    while len(out) < n and page <= 25:
        p = {"taxon_id": tid, "photos": "true", "quality_grade": "research", "per_page": 200, "page": page, "order_by": "votes"}
        if INAT_CC_ONLY:
            p["photo_license"] = "cc0,cc-by,cc-by-nc,cc-by-sa,cc-by-nd,cc-by-nc-sa,cc-by-nc-nd"
        res = get_json("https://api.inaturalist.org/v1/observations", p).get("results", [])
        if not res:
            break
        for o in res:
            for ph in o.get("photos", [])[:1]:                          # 관찰 하나당 첫 사진만 (비슷한 연사 방지)
                out.append(item(ph["url"].replace("/square.", "/large."), ph.get("license_code") or "all-rights-reserved",
                                ph.get("attribution"), o.get("uri")))
        page += 1
        time.sleep(1)                                                    # iNaturalist 권장: 초당 1회 이하
    return out[:n]

def src_gbif(q, n):
    key = get_json("https://api.gbif.org/v1/species/match", {"name": q}).get("usageKey")
    if not key:
        return []
    out, off = [], 0
    while len(out) < n:
        r = get_json("https://api.gbif.org/v1/occurrence/search", {"taxonKey": key, "mediaType": "StillImage",
                     "basisOfRecord": "HUMAN_OBSERVATION", "limit": 300, "offset": off})
        for o in r.get("results", []):
            for m in o.get("media", [])[:1]:
                u = m.get("identifier")
                if u and m.get("type", "StillImage") == "StillImage":
                    if "inaturalist" in u:
                        u = u.replace("/original.", "/large.")
                    out.append(item(u, m.get("license"), m.get("rightsHolder") or m.get("creator"),
                                    f"https://www.gbif.org/occurrence/{o.get('key')}"))
        if r.get("endOfRecords", True) or not r.get("results"):
            break
        off += 300
    return out[:n]

def src_commons(q, n):
    out, cont = [], {}
    while len(out) < n:
        r = get_json("https://commons.wikimedia.org/w/api.php",
                     {"action": "query", "format": "json", "generator": "search", "gsrsearch": f"filetype:bitmap {q}",
                      "gsrnamespace": 6, "gsrlimit": 50, "prop": "imageinfo", "iiprop": "url|extmetadata", "iiurlwidth": 1600, **cont},
                     headers={"User-Agent": API_UA})
        for p in r.get("query", {}).get("pages", {}).values():
            ii = (p.get("imageinfo") or [{}])[0]
            md = ii.get("extmetadata", {})
            u = ii.get("thumburl") or ii.get("url")
            if u:
                out.append(item(u, md.get("LicenseShortName", {}).get("value"),
                                re.sub(r"<[^>]+>", "", md.get("Artist", {}).get("value", "")), ii.get("descriptionurl")))
        if "continue" not in r:
            break
        cont = r["continue"]
        time.sleep(1)
    return out[:n]

def src_openverse(q, n):
    out, page = [], 1
    while len(out) < n and page <= 12:                                  # 비로그인 요청은 쪽당 20장, 약 240장까지
        r = get_json("https://api.openverse.org/v1/images/", {"q": q, "page_size": 20, "page": page}, headers={"User-Agent": API_UA})
        res = r.get("results", [])
        for x in res:
            out.append(item(x.get("url"), f"{x.get('license', '')} {x.get('license_version', '')}".strip(), x.get("creator"), x.get("foreign_landing_url")))
        if not res or page >= r.get("page_count", 0):
            break
        page += 1
        time.sleep(1)
    return out[:n]

def src_flickr(q, n):
    if not FLICKR_API_KEY:
        return []
    out, page = [], 1
    while len(out) < n:
        r = get_json("https://api.flickr.com/services/rest/",
                     {"method": "flickr.photos.search", "api_key": FLICKR_API_KEY, "text": q, "content_types": 0, "media": "photos",
                      "extras": "url_b,url_c,license,owner_name", "per_page": 250, "page": page, "sort": "relevance",
                      "format": "json", "nojsoncallback": 1})
        ps = r.get("photos", {}).get("photo", [])
        for p in ps:
            u = p.get("url_b") or p.get("url_c")
            if u:
                out.append(item(u, f"flickr-license-{p.get('license')}", p.get("ownername"), f"https://www.flickr.com/photos/{p['owner']}/{p['id']}"))
        if not ps or page >= r.get("photos", {}).get("pages", 0):
            break
        page += 1
    return out[:n]

# ---------- 검색 엔진 ----------
def src_naver(q, n):
    try:
        r = HTTP.get("https://search.naver.com/search.naver", params={"where": "image", "query": q}, timeout=20)
    except requests.RequestException:
        return []
    urls = dict.fromkeys(u.replace("\\/", "/") for u in re.findall(r'"originalUrl"\s*:\s*"([^"]+)"', r.text))
    return [item(u) for u in urls][:n]

def src_bing(q, n):
    out, seen = [], set()
    for first in range(0, n + 35, 35):
        try:
            r = HTTP.get("https://www.bing.com/images/async", timeout=20,
                         params={"q": q, "first": first, "count": 35, "adlt": "strict", "qft": "+filterui:photo-photo"})
        except requests.RequestException:
            break
        new = 0
        for m in re.findall(r'\bm="([^"]+)"', r.text):
            try:
                d = json.loads(html.unescape(m))
            except ValueError:
                continue
            u = d.get("murl")
            if u and u not in seen:
                seen.add(u); new += 1
                out.append(item(u, page=d.get("purl")))
        if not new or len(out) >= n:
            break
        time.sleep(0.5)
    return out[:n]

def src_ddg(q, n):
    from ddgs import DDGS
    out, seen = [], set()
    for page in range(1, 6):
        try:
            res = DDGS().images(q, max_results=100, page=page, type_image="photo", safesearch="moderate")
        except Exception:
            break
        new = [r for r in res if r.get("image") and r["image"] not in seen]
        for r in new:
            seen.add(r["image"])
            out.append(item(r["image"], page=r.get("url")))
        if not new or len(out) >= n:
            break
        time.sleep(1)
    return out[:n]

def src_baidu(q, n):
    from icrawler import ImageDownloader
    from icrawler.builtin import BaiduImageCrawler
    got = []
    class UrlOnly(ImageDownloader):                                     # 주소만 모으고 다운로드는 아래 공통 함수로
        def download(self, task, default_ext, timeout=5, max_retry=3, overwrite=False, **kwargs):
            got.append(task["file_url"])
    try:
        BaiduImageCrawler(downloader_cls=UrlOnly, storage={"root_dir": "/tmp/_baidu"}, log_level=logging.CRITICAL).crawl(keyword=q, max_num=n)
    except Exception:
        pass
    return [item(u, page="https://image.baidu.com") for u in dict.fromkeys(got)][:n]

# (폴더 접두 번호 = 중복일 때 남길 우선순위, 함수, 사용할 검색어 종류)
SOURCE_DEFS = {
    "inat":      (1, src_inat,      ["taxa"]),
    "gbif":      (2, src_gbif,      ["taxa"]),
    "commons":   (3, src_commons,   ["taxa", "en"]),
    "openverse": (4, src_openverse, ["taxa", "en"]),
    "flickr":    (5, src_flickr,    ["taxa", "en"]),
    "naver":     (6, src_naver,     ["ko"]),
    "bing":      (7, src_bing,      ["en"]),
    "ddg":       (8, src_ddg,       ["en"]),
    "baidu":     (9, src_baidu,     ["zh"]),
}

# ---------- 공통 다운로드 ----------
MAGIC = [(b"\xff\xd8\xff", ".jpg"), (b"\x89PNG", ".png"), (b"GIF8", ".gif"), (b"BM", ".bmp")]

def url_key(u):
    m = re.search(r"inaturalist[^/]*/photos/(\d+)/", u)                # iNaturalist 사진은 GBIF와 주소가 달라도 같은 사진
    return f"inat:{m.group(1)}" if m else u

def fetch(u, path_noext):
    try:
        r = HTTP.get(u, timeout=20)
    except requests.RequestException:
        return None
    b = r.content
    if r.status_code != 200 or len(b) < 8000:
        return None
    ext = next((e for sig, e in MAGIC if b.startswith(sig)), ".webp" if b[8:12] == b"WEBP" else None)
    if not ext:
        return None                                                      # HTML 오류 페이지 등
    with open(path_noext + ext, "wb") as fp:
        fp.write(b)
    return path_noext + ext

def save_manifest():
    with _lock:
        json.dump(MANIFEST, open(MANIFEST_PATH, "w", encoding="utf-8"), ensure_ascii=False)

def collect(group, src, qi, query, items):
    prio = SOURCE_DEFS[src][0]
    out_dir = os.path.join(RAW, group, f"{prio}_{src}")
    os.makedirs(out_dir, exist_ok=True)
    todo = []
    for k, it in enumerate(items):
        key = url_key(it["url"])
        if key in MANIFEST:                                               # 이미 받았거나 다른 경로에서 받은 사진
            continue
        MANIFEST[key] = None                                              # 같은 실행 안에서의 중복 방지
        todo.append((key, it, os.path.join(out_dir, f"q{qi:02d}_{k:04d}")))
    def work(t):
        key, it, p = t
        path = fetch(it["url"], p)
        MANIFEST[key] = {**it, "path": path, "group": group, "source": src, "query": query} if path else {"path": None}
        return path is not None
    with ThreadPoolExecutor(DOWNLOAD_THREADS) as ex:
        ok = sum(ex.map(work, todo))
    save_manifest()
    return len(items), len(todo), ok

MANIFEST = {k: v for k, v in MANIFEST.items() if v and v.get("path") and os.path.exists(v["path"])}   # 실패·삭제된 항목은 다시 시도
counts, misses, skipped = {}, {}, set()
for group, spec in GROUPS.items():
    is_class = group in CFG["classes"]
    for src, conf in SOURCES.items():
        if not conf["on"] or src in skipped:
            continue
        _, fn, kinds = SOURCE_DEFS[src]
        queries = list(dict.fromkeys(q for kind in kinds for q in spec.get(kind, [])))
        n = conf["max"] if is_class else max(10, int(conf["max"] * NEG_RATIO))
        for qi, q in enumerate(queries):
            if src in skipped:
                break
            t0 = time.time()
            try:
                items = fn(q, n)
            except Exception as e:
                print(f"  [{group}] {src} '{q}': 실패 ({type(e).__name__}: {e})")
                items = []
            misses[src] = 0 if items else misses.get(src, 0) + 1
            if misses[src] >= 3:                                         # 연속 3번 결과가 없으면 막힌 것으로 보고 건너뜀
                skipped.add(src)
                print(f"  ※ {src}: 연속으로 결과가 없어 이번 실행에서는 건너뜁니다 (나중에 셀 3을 다시 실행하면 재시도)")
            found, new, ok = collect(group, src, qi, q, items)
            counts[(group, src)] = counts.get((group, src), 0) + ok
            print(f"  [{group}] {src:9s} '{q}': 주소 {found} → 새 사진 {new} → 저장 {ok}  ({time.time() - t0:.0f}초)")

import pandas as pd
table = pd.Series(counts, dtype=int).unstack(fill_value=0) if counts else pd.DataFrame()
if not table.empty:
    table["합계"] = table.sum(axis=1)
print("\n이번 실행에서 새로 받은 사진 수 (그룹 × 경로)")
display(table)
print("누적 저장:", sum(1 for v in MANIFEST.values() if v and v.get("path")), "장")

In [ ]:
# ===== 셀 4: 자동 정제 (깨진 파일 / 작은 사진 / 극단적 비율 / 중복 제거) =====
import glob, random, shutil
from PIL import Image, ImageOps
import imagehash

MIN_SIDE   = 320    # 짧은 변이 이보다 작으면 제외
MAX_ASPECT = 3.0    # 가로세로 비율이 이보다 크면 제외 (배너·파노라마)
MAX_SIDE   = 1600   # 업로드 용량을 줄이기 위해 긴 변을 이 크기로 축소
HASH_DIST  = 6      # 지각 해시 거리 ≤ 이 값이면 중복으로 판단
KEEP_MAX   = None   # 그룹당 최대 장수 (예: 600). None이면 전부 남김. 라벨링 시간이 부족하면 줄이기

EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
shutil.rmtree(CLEAN, ignore_errors=True)
seen_hashes, CLEAN_MAP, stats = [], [], []

def is_duplicate(h):
    return any((h ^ s).bit_count() <= HASH_DIST for s in seen_hashes)

groups = list(GROUPS)
for group in groups:
    src, dst = os.path.join(RAW, group), os.path.join(CLEAN, group)
    os.makedirs(dst, exist_ok=True)
    kept_files, bad, small, dup = [], 0, 0, 0
    # 폴더 이름의 번호 순(1_inat → 9_baidu)으로 읽으므로, 중복이면 품질이 좋은 경로의 사진이 남음
    for p in sorted(glob.glob(os.path.join(src, "**", "*"), recursive=True)):
        if not p.lower().endswith(EXTS):
            continue
        try:
            with Image.open(p) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
        except Exception:
            bad += 1; continue
        w, h = im.size
        if min(w, h) < MIN_SIDE or max(w, h) / min(w, h) > MAX_ASPECT:
            small += 1; continue
        hv = int(str(imagehash.phash(im)), 16)
        if is_duplicate(hv):
            dup += 1; continue
        seen_hashes.append(hv)
        kept_files.append(p)
    if KEEP_MAX and len(kept_files) > KEEP_MAX:
        kept_files = sorted(random.Random(0).sample(kept_files, KEEP_MAX))
    for k, p in enumerate(kept_files, 1):
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((MAX_SIDE, MAX_SIDE))
            out = os.path.join(dst, f"{group}_{k:04d}.jpg")
            im.save(out, "JPEG", quality=92)
        CLEAN_MAP.append([out, p])
    stats.append((group, len(kept_files), bad, small, dup))

print(f"{'그룹':<22}{'남음':>6}{'깨짐':>6}{'작음/비율':>10}{'중복':>6}")
for g, k, b, s, d in stats:
    print(f"{g:<22}{k:>6}{b:>6}{s:>10}{d:>6}")

In [ ]:
# ===== 셀 5: 검수용 미리보기 (번호 확인 후 셀 6에 적기) =====
from PIL import ImageDraw
from IPython.display import display

def contact_sheets(group, cols=8, rows=6, thumb=170):
    os.makedirs(REVIEW, exist_ok=True)
    files = sorted(glob.glob(os.path.join(CLEAN, group, "*.jpg")))
    per = cols * rows
    for start in range(0, len(files), per):
        sheet = Image.new("RGB", (cols * thumb, rows * (thumb + 20)), "white")
        draw = ImageDraw.Draw(sheet)
        for k, f in enumerate(files[start:start + per]):
            with Image.open(f) as im:
                im.thumbnail((thumb, thumb))
                x, y = (k % cols) * thumb, (k // cols) * (thumb + 20)
                sheet.paste(im, (x + (thumb - im.width) // 2, y))
            num = int(os.path.basename(f).rsplit("_", 1)[1].split(".")[0])
            draw.text((x + 4, y + thumb + 3), str(num), fill="black")
        path = os.path.join(REVIEW, f"{group}_p{start // per + 1}.jpg")
        sheet.save(path, quality=85)
        print(path); display(sheet)

for group in groups:
    print(f"\n===== {group} =====")
    contact_sheets(group)

In [ ]:
# ===== 셀 6: 잘못된 사진 삭제 =====
# 미리보기 번호를 적고 실행하세요. 예) "giant_hornet": [3, 17, 42]
DELETE = {
    # "honeybee": [],
    # "giant_hornet": [],
    # "yellow_legged_hornet": [],
    # "hard_negative": [],
    # "background": [],
}

removed = 0
for group, nums in DELETE.items():
    for n in nums:
        f = os.path.join(CLEAN, group, f"{group}_{n:04d}.jpg")
        if os.path.exists(f):
            os.remove(f); removed += 1
print("삭제:", removed, "장")
for group in groups:
    print(f"  {group:<18}{len(glob.glob(os.path.join(CLEAN, group, '*.jpg'))):>5}장")

In [ ]:
# ===== 셀 7: 출처 기록 저장 + 내려받기 =====
import csv
SAVE_TO_DRIVE = False   # True면 Google Drive(MyDrive/beeguard)에도 복사 (zip이 커서 브라우저 다운로드가 끊길 때)

meta_of = {v["path"]: v for v in MANIFEST.values() if v and v.get("path")}
with open("/content/sources.csv", "w", newline="", encoding="utf-8-sig") as fp:
    wr = csv.writer(fp)
    wr.writerow(["clean_file", "group", "source", "query", "source_url", "page_url", "license", "author"])
    for out, raw in CLEAN_MAP:
        if os.path.exists(out):              # 검수에서 지운 파일은 제외
            m = meta_of.get(raw, {})
            wr.writerow([os.path.relpath(out, CLEAN), m.get("group", ""), m.get("source", ""), m.get("query", ""),
                         m.get("url", ""), m.get("page", ""), m.get("license", ""), m.get("author", "")])

shutil.make_archive("/content/dataset_clean", "zip", CLEAN)
size_mb = os.path.getsize("/content/dataset_clean.zip") / 2**20
print(f"dataset_clean.zip ({size_mb:.0f} MB), sources.csv 생성 완료")
print(pd.read_csv("/content/sources.csv", encoding="utf-8-sig").groupby(["group", "source"]).size().unstack(fill_value=0))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/beeguard", exist_ok=True)
    for f in ["/content/dataset_clean.zip", "/content/sources.csv"]:
        shutil.copy(f, "/content/drive/MyDrive/beeguard/")
    print("Google Drive의 MyDrive/beeguard 에 복사했습니다.")
try:
    from google.colab import files
    files.download("/content/dataset_clean.zip")
    files.download("/content/sources.csv")
except ImportError:
    pass